# Polarization of Light

### An interactive introduction

Light is a wave, and as it travels its electric field oscillates in some
direction. That direction is the **polarization** and it gives us information that brightness
alone does not carry. In astronomy it is the most direct probe available of magnetic fields.

This is why the Event Horizon Telescope's image of M87 is usually published with
short lines drawn across the bright ring. Those lines are polarization, and they
trace the magnetic field threading the plasma orbiting a supermassive black
hole, the same field thought to launch a jet thousands of light years long. Nothing
else in the picture says anything about that field.

The notebook builds the idea up from nothing: one wave, then a telescope trying
to record it, then the messy reality of a whole glowing plasma. Every figure is interactive, and each one exists to build the 
intuition behind a concept before introducing an equation for it.

**What it covers**

1. How two amplitudes and a phase offset describe any polarization state, and
   what shape the field actually traces, which is the polarization ellipse;
2. Why a telescope can only ever record two projections of that shape, and why
   different telescopes chose different pairs: **linear** $(X, Y)$ against
   **circular** $(R, L)$;
3. How a chaotic ensemble of emitters gives *partial* polarization, and why the
   four Stokes parameters $I, Q, U, V$ are the honest way to describe it;
4. Where astrophysical polarization comes from (synchrotron emission), and what
   happens to it on the way to us, through Faraday rotation;
5. What a polarized image is, and why the structure has to be *resolved* before
   it can be measured at all.

**Prerequisites:** complex numbers. No radio astronomy assumed.

---

> **How to use this notebook.** Drag the sliders! Every figure recomputes as you
> move them. None of the code is written here: the physics is in
> `scripts/polarization.py` and `scripts/images.py`, the figures in
> `scripts/figures_polarization.py`, and the slider wiring in
> `scripts/interactive_polarization.py`, so each part can be read on its own.
> Interactivity needs a running kernel. A static render shows the figures, but
> the sliders will not move.

In [1]:
# Every function this notebook calls lives in ../scripts.
import sys
from pathlib import Path

SCRIPTS = Path.cwd() / "scripts"
if not SCRIPTS.is_dir():
    SCRIPTS = Path.cwd().parent / "scripts"
sys.path.append(str(SCRIPTS))
print("importing from", SCRIPTS)

import figures_polarization as figs
import images
import interactive_polarization as explore
import numpy as np
import polarization as pol

importing from /mnt/c/Users/alexw/OneDrive - University of Toronto/SUMMER_26/EHT/CSC494-report/scripts


## 1. Light as a wave

Light travels as an electromagnetic wave. The vibration is **transverse**, meaning that
the electric field points *across* the direction of travel, never along it. So
if a wave is coming straight at you out of the page, its field lives in the
two-dimensional plane of the page.

That vibration comes in two recognisable ways:

- **Linear polarization** : the field oscillates back and forth along a straight
  line, like a guitar string plucked up-and-down or side-to-side.
- **Circular polarization** : the field doesn't stay on a line at all. It keeps
  the same length and rotates, sweeping out a corkscrew as the wave travels.

Both are the same physics with one number changed. 

The field is described with two
perpendicular oscillations, directions $x$ and $y$, each with its
own amplitude. Let the $y$ one lag behind the $x$ one by a phase $\delta$ and we get:

$$\mathbf{E}(t) = \mathrm{Re}\left[\begin{pmatrix} a_x \\ a_y e^{i\delta}
\end{pmatrix} e^{i\omega t}\right]$$

That complex pair is called a **Jones vector**, and it is the complete
description of one wave's polarization. Three numbers: two amplitudes and the
phase offset between them.

The phase offset is the interesting one. Drag it and watch a flat wave balloon
into a corkscrew:

In [2]:
explore.wave_3d_explorer()

- At **$\delta = 0$** the two components rise and fall together. Their sum stays
  in one plane, and the wave is a flat sinusoid: **linearly polarized**.
- At **$\delta = \pm90°$ with equal amplitudes**, one component peaks while the
  other is passing through zero. Their sum never shrinks and never stays in a
  plane: a **circular** corkscrew. The sign of $\delta$ chooses which way it
  turns.
- Anything in between is a stretched corkscrew, an ellipse in cross-section.

### The shape it traces: the polarization ellipse

Stop travelling with the wave and stand at one point in space. Watch the tip of
the electric field vector over a single cycle. Where does it go?

It has no choice but to trace an **ellipse**. Both components are sinusoids at
the same frequency, so the tip is at
$(a_x\cos\omega t,\; a_y\cos(\omega t - \delta))$, which is the parametric
equation of an ellipse for every choice of $a_x$, $a_y$ and $\delta$. The two
flavours above are just its degenerate limits: squash the ellipse flat and it is
a line (linear); make it perfectly round and it is a circle (circular).

This **polarization ellipse** is the single most useful picture in the subject,
because everything measurable about one wave is a property of it:

| property of the ellipse | what it is called | what it tells you |
|---|---|---|
| the tilt of its long axis | **EVPA**, $\chi$ | the direction of vibration; *this is what the tick marks on the M87 image show* |
| how fat it is (short axis / long axis) | ellipticity | how circular the light is; zero for a line, one for a circle |
| which way it is traced | handedness | clockwise or counter-clockwise, as seen by the observer |
| its overall size | intensity | how much light there is |

The figure below draws it directly. Left is the ellipse being traced, with the
instantaneous field vector marked; right is the same information as two
oscillations with a phase offset, which is the view the algebra above describes.
The preset buttons walk through the special cases.

In [3]:
explore.wave_explorer()

Watch the tilt as you change the *ratio* of the amplitudes with $\delta = 0$: a
tall thin wave is vertical polarization, an equal mix is polarization at $45°$.
The tilt is set by the amplitudes; the fatness is set by the phase.

One warning that will matter later: an ellipse has **no arrowhead**. A tilt of
$0°$ and a tilt of $180°$ describe exactly the same shape, so EVPA is only ever
defined modulo $180°$. Every mysterious $90°$ flip in polarimetry traces back to
that fact.

## 2. Telescopes are polarized sunglasses

Nothing in a telescope can record that 3D shape directly. What a receiver
actually has is a pair of **feeds** (two little antennas at the focus of the dish)
and each feed responds to the projection of the incoming field onto one
particular direction, producing a voltage.

So a telescope is a pair of polarized sunglasses, and here the story gets
awkward, because there's two different kinds:

- **Circular feeds $(R, L)$** respond to light spiralling right or left. The Event Horizon Telescope was built this way.
- **Linear feeds $(X, Y)$** respond to light oscillating along two perpendicular
  straight directions.

Neither is more correct. They are two coordinate systems for the same
two-dimensional space, related by a change of basis. The standard convention in
radio astronomy (IAU) is

$$\begin{bmatrix} R \\ L \end{bmatrix} = \frac{1}{\sqrt{2}}
\begin{bmatrix} 1 & i \\ 1 & -i \end{bmatrix}
\begin{bmatrix} X \\ Y \end{bmatrix}$$

The matrix is unitary, so no information is created or destroyed but the
*numbers* recorded are completely different. Watch what a linear
receiver and an circular receiver report for the very same wave:

In [4]:
explore.basis_explorer()

Press **right circular**. The circular feeds put *all* of the light into one
number and nothing into the other, which is a clean and obvious answer. The linear feeds
split it exactly evenly and hide the entire story in a $90°$ phase difference
between two equal amplitudes. Press **linear, horizontal** and it happens in
reverse.

The same thing in code, since it is one matrix multiply:

In [5]:
field = pol.jones_vector(amp_x=1 / np.sqrt(2), amp_y=1 / np.sqrt(2), delta=np.deg2rad(-90))
print("linear feeds   (E_X, E_Y) =", np.round(field, 3))
print("circular feeds (E_R, E_L) =", np.round(pol.lin_to_circ(field), 3))
print("\nAll of the light is in R: this wave is purely right-circular.")

linear feeds   (E_X, E_Y) = [0.707+0.j    0.   -0.707j]
circular feeds (E_R, E_L) = [1.+0.j 0.-0.j]

All of the light is in R: this wave is purely right-circular.


As long as every dish in an array speaks the same language, this can be ignored, we only convert once at the start and carry on. **Mixed polarization** is what happens
when they don't, and it is the subject of the notebooks that follow.

## 3. Stokes parameters

Everything so far described a single, perfectly polarized wave. Astronomical sources are not like that. A black hole's accretion disk is an enormous number of independent
emitters, each radiating its own wave with its own phase and its own direction. 
When these millions of independent waves hit the telescope at the exact same time, their different vibrations fight against each other.

Most of the random phases cancel each other out. What
survives the averaging is intensity-like quantities, and the consequence is that
real light is only **partially** polarized: part of it agrees on a direction, the
rest averages away. Drag the spread of the emitters' angles and watch order
disappear:

In [6]:
explore.depolarization_explorer()

The right-hand panel is the punchline: the emitters have to agree to within a few
tens of degrees for any appreciable polarization to survive. The EHT measures
linear polarization fractions of order 10–20% in M87, and *that number is a
measurement of how ordered the magnetic field is*.

So the single-wave description is not enough. What we need are quantities that a
real receiver can measure and that *add* over an incoherent pile of waves. There
are exactly four, the **Stokes parameters**, and they behave like four filters
laid over the light:

$$I = \langle|E_X|^2\rangle + \langle|E_Y|^2\rangle, \qquad
  Q = \langle|E_X|^2\rangle - \langle|E_Y|^2\rangle, \qquad
  U = 2\,\mathrm{Re}\langle E_X E_Y^*\rangle, \qquad
  V = 2\,\mathrm{Im}\langle E_X E_Y^*\rangle$$

- **$I$** is total brightness, polarized or not. The ordinary picture.
- **$Q$** and **$U$** are the linear part. $Q$ is the excess of horizontal over
  vertical, $U$ the excess of $45°$ over $135°$.
- **$V$** is the circular part: right-handed corkscrews minus left-handed ones.

$I$ is a sum of squares and so always positive. The other three are
*differences*, so they take either sign and can vanish even when there is plenty
of light. One bound ties them together:

$$I^2 \ge Q^2 + U^2 + V^2$$

Light can be at most 100% polarized; states outside that bound do not exist.

That leaves one obvious question. If the direction of vibration is a single
angle, why does it take *two* numbers to record it? Drag the EVPA slider and read
the answer off the bars.

In [5]:
explore.stokes_explorer()

Sweeping the EVPA from $0°$ to $90°$ sends $Q$ from positive through zero to
negative while $U$ rises and falls. The two behave exactly like a $\cos$ and a
$\sin$. They are not two independent facts about the light; they are one angle,
stored the only way a linear detector can store it. That is why $Q$ and $U$ maps
are always published side by side, and it is what the two derived quantities
unpack:

$$p = \frac{\sqrt{Q^2 + U^2}}{I}, \qquad \chi = \tfrac{1}{2}\arctan\frac{U}{Q}$$

The **polarization fraction** $p$ says *how* polarized the light is: $1$ is fully
ordered, $0$ completely scrambled. The **EVPA** $\chi$ says in *which* direction,
and it is exactly the tilt of the polarization ellipse from Section 1, now
recovered from two measurable numbers. Note the factor of one half: two full
turns of $(Q, U)$ correspond to one turn of $\chi$, which is the $180°$ ambiguity
of Section 1 showing up in the algebra.

### Optional: the map of every state

There is a tidy picture of the whole space. Plot $(Q, U, V)/I$ in three
dimensions and every possible polarization state is a point inside the unit ball:
the surface is fully polarized light, the centre is completely unpolarized, the
equator is linear and the poles are circular. It is called the **Poincaré
sphere**, and it makes the $180°$ ambiguity geometric: walking the EVPA through
$180°$ carries the point all the way around the equator.

In [13]:
explore.poincare_explorer()

> **A note on conventions.** Every formula above matches `eht-imaging`'s, which
> are documented in `docs/polarization_conventions.md`: the IAU /
> Hamaker–Bregman–Sault choice with an $e^{+i\omega t}$ time dependence, in which
> $V > 0$ means right-handed. Flip the time convention and $V$ changes sign; the
> correlator formulas used in the software carry an extra factor of $\tfrac12$
> from how feed voltages are normalised. This sounds pedantic and is not:
> `scripts/tests/test_polarization.py` pins every canonical state against
> hand-derived values and cross-checks the linear and circular formulas against
> each other, because a sign error here is invisible until it has silently
> corrupted an image.

## 4. Where the polarization comes from

So light can be polarized, and four numbers describe how. Why does the sky
bother?

The relevant mechanism near a black hole is **synchrotron emission**. Electrons
moving at nearly the speed of light spiral around magnetic field lines, and an
accelerating charge radiates. Because the acceleration is mostly perpendicular to
the field, so is the emitted electric field: for optically thin emission, the
**EVPA comes out perpendicular to the magnetic field projected on the sky**.

That one sentence is why polarimetry gets telescope time. An EVPA map is a
magnetic field map, rotated by $90°$.

In [9]:
explore.synchrotron_explorer()

The dashed line is the magnetic field; the black ticks are what a telescope
measures. Note the *optically thick* toggle: when the emission is thick the
relationship flips by another $90°$, so you have to know which regime you are in
before claiming a field geometry from a tick map.

## 5. What happens on the way out

Between the emitting plasma and us sits more magnetised plasma, and it rotates
the plane of linear polarization as the light passes through. That is **Faraday
rotation**, and it is chromatic, growing as the square of the wavelength:

$$\chi_{\rm obs} = \chi_0 + \mathrm{RM}\,\lambda^2$$

The **rotation measure** RM is proportional to the line-of-sight magnetic field
times the electron density, so it is another probe of the field, but only if you
observe at more than one frequency, because a single $\lambda$ cannot separate
$\chi_0$ from $\mathrm{RM}\lambda^2$.

In [ ]:
explore.faraday_explorer()

## 6. What we actually want: an image

Put it together. The target is not four numbers, it is four *images*: $I$, $Q$,
$U$ and $V$ as functions of position on the sky. The conventional way to draw
them is Stokes $I$ as a brightness map with ticks on top, each tick pointing
along the local EVPA (that is, along the tilt of the local polarization
ellipse), with its length showing how strongly polarized that spot is.

Below is a synthetic M87-like ring: a bright asymmetric annulus about 42
microarcseconds across, with a spiral EVPA pattern of the kind the EHT actually
measured. The **pitch angle** slider sweeps the pattern from radial to azimuthal,
which corresponds to different magnetic field geometries: a purely toroidal
field gives one, a poloidal field the other, and the measured spiral in between
is what argued for a dynamically important field in M87.

In [ ]:
explore.image_explorer()

Read the subtitle as you drag. Every pixel on that ring is 25% polarized, but the
ring as a whole nets barely 1%, because opposite sides have nearly perpendicular
EVPAs and cancel. A single-dish measurement would see essentially nothing.

**That cancellation is the argument for the entire EHT.** The field geometry
cannot be learned from an unresolved polarization measurement; the structure has
to be resolved.

Here are the four Stokes images behind that figure. $I$ gets a brightness map;
$Q$, $U$ and $V$ are signed, so they get a diverging map with a neutral midpoint
where blue and red mean opposite sign:

In [ ]:
ring = images.polarized_ring(npix=128, fov_uas=100.0, p_lin=0.25, pitch_deg=45.0)
totals = images.image_stokes_totals(ring)
print(f"total flux       I = {totals['I']:.3f} Jy")
print(f"per pixel        p = {totals['p_lin_mean']:.1%}")
print(f"net (unresolved) p = {totals['p_lin_net']:.2%}   <- the cancellation")

figs.stokes_panels_figure(ring)

total flux       I = 1.000 Jy
per pixel        p = 25.0%
net (unresolved) p = 0.00%   <- the cancellation


Notice that $Q$ and $U$ change sign around the ring while $I$ never does, and
that $V$ is identically zero here because the model was built with no circular
polarization. Real circular polarization in M87 sits at the few-tenths-of-a-percent
level, small enough that instrumental leakage is a serious contender for
anything you think you have measured.

## Recap

- A wave's polarization is two amplitudes and a **phase offset**. Over one cycle
  the field tip traces the **polarization ellipse**, whose tilt is the EVPA and
  whose fatness is the ellipticity; linear and circular light are its two
  degenerate limits.
- A telescope cannot see that ellipse. It records **two projections** of it, and
  the array is split between **linear $(X, Y)$** and **circular $(R, L)$** feeds:
  the same light, two incompatible sets of numbers.
- Real emission is an incoherent pile of waves, so it is only **partially**
  polarized, and the additive, measurable description is $I$, $Q$, $U$, $V$, with
  $p = \sqrt{Q^2+U^2}/I$ and $\chi = \frac12\arctan(U/Q)$.
- For optically thin synchrotron emission the EVPA is **perpendicular to the
  projected magnetic field**: an EVPA map is a B-field map.
- Faraday rotation adds $\mathrm{RM}\,\lambda^2$ on the way out, which is why the
  measurement has to be made per frequency channel.
- What we want is a polarized **image**, because the net polarization of an
  unresolved source largely cancels.

**Next:** [The Event Horizon Telescope](02_the_eht.ipynb). That ring is 42
microarcseconds across, and no single telescope on Earth can resolve it. So the
array fakes one the size of the planet.